# Combining Datasets: Merge and Join

One essential feature offered by Pandas is its in-memory join and merge operations.
This is similar to join in databases.
The main interface for this is the ``pd.merge`` .


In [1]:
# import the libraries
import pandas as pd
import numpy as np 

## Categories of Joins

The ``pd.merge()`` function implements a number of types of joins: the *one-to-one*, *many-to-one*, and *many-to-many* joins.
All three types of joins are accessed via an identical call to the ``pd.merge()`` interface; the type of join performed depends on the form of the input data.
Here we will show simple examples of the three types of merges, and discuss detailed options further below.

### One-to-one joins

Perhaps the simplest type of merge expresion is the one-to-one join.
Example- consider the following two ``DataFrames`` which contain information on several employees in a company:

In [4]:
d = {'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'dept': ['Accounting', 'Engineering', 'Engineering', 'HR']}
type(d)

dict

In [12]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'dept': ['Accounting', 'Engineering', 'Engineering', 'HR']})


df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2015, 2018, 2019, 2017]})

df1

,employee,dept
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR


To combine this information into a single ``DataFrame``, we can use the ``pd.merge()`` function:

In [13]:
df3 = pd.merge(df1, df2)
df3

,employee,dept,hire_date
0,Bob,Accounting,2018
1,Jake,Engineering,2019
2,Lisa,Engineering,2015
3,Sue,HR,2017


In [10]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   employee   4 non-null      object
 1   dept       4 non-null      object
 2   hire_date  4 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 128.0+ bytes


The ``pd.merge()`` function recognizes that each ``DataFrame`` has an "employee" column, and automatically joins using this column as a key.
The result of the merge is a new ``DataFrame`` that combines the information from the two inputs.
Notice that the order of entries in each column is not necessarily maintained: in this case, the order of the "employee" column differs between ``df1`` and ``df2``, and the ``pd.merge()`` function correctly accounts for this.
Also merge in general discards the index, except in the special case of merges by index.

### Many-to-one joins

Many-to-one joins are joins in which one of the two key columns contains duplicate entries.
For the many-to-one case, the resulting ``DataFrame`` will preserve those duplicate entries as appropriate.
Consider the following example of a many-to-one join:

In [17]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   employee   4 non-null      object
 1   dept       4 non-null      object
 2   hire_date  4 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 128.0+ bytes


In [15]:
df4 = pd.DataFrame({'dept': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})

#df3 # has dept entry
#df4 #has dept entry also, so a join is possible
pd.merge(df3,df4)

,employee,dept,hire_date,supervisor
0,Bob,Accounting,2018,Carly
1,Jake,Engineering,2019,Guido
2,Lisa,Engineering,2015,Guido
3,Sue,HR,2017,Steve


In [12]:
pd.merge(df3, df4)

,employee,dept,hire_date,supervisor
0,Bob,Accounting,2018,Carly
1,Jake,Engineering,2019,Guido
2,Lisa,Engineering,2015,Guido
3,Sue,HR,2017,Steve


The resulting ``DataFrame`` has an aditional column with the "supervisor" information, where the information is repeated in one or more locations as required by the inputs.

### Many-to-many joins

Many-to-many joins are a bit confusing conceptually, but are nevertheless well defined.
If the key column in both the left and right array contains duplicates, then the result is a many-to-many merge.

Consider a ``DataFrame`` showing one or more skills associated with a particular dept.
By performing a many-to-many join, we can recover the skills associated with any individual person:

In [16]:
df5 = pd.DataFrame({'dept': ['Accounting', 'Accounting','Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux','spreadsheets', 'organisation']})

#df5
pd.merge(df1,df5)

,employee,dept,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organisation


## Specification of the Merge Key

Default behavior of ``pd.merge()``: looks for one or more matching column names between the two inputs.
Often the column names will not match that well, and ``pd.merge()``  provides a way to dictate the merge column.

### The ``on`` keyword

Specify the name of the merge column using the ``on`` keyword, which takes a column name or a list of column names:

In [7]:
pd.merge(df1, df2, on = "employee")

,employee,dept,hire_date
0,Bob,Accounting,2018
1,Jake,Engineering,2019
2,Lisa,Engineering,2015
3,Sue,HR,2017


This option works only if both the left and right ``DataFrames`` have the specified column name.

### The ``left_on`` and ``right_on`` keywords

At times you may wish to merge two datasets with different column names; for example, we may have a dataset in which the employee name is labeled as "name" rather than "employee".
In this case, we can use the ``left_on`` and ``right_on`` keywords to specify the two column names:

In [17]:
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})

pd.merge(df1, df3, left_on="employee", right_on = "name")

,employee,dept,name,salary
0,Bob,Accounting,Bob,70000
1,Jake,Engineering,Jake,80000
2,Lisa,Engineering,Lisa,120000
3,Sue,HR,Sue,90000


The result has a redundant column that we can drop if desired–for example, by using the ``drop()``.

In [18]:
pd.merge(df1, df3, left_on="employee", right_on="name").drop('name', axis = 1)

,employee,dept,salary
0,Bob,Accounting,70000
1,Jake,Engineering,80000
2,Lisa,Engineering,120000
3,Sue,HR,90000


### The ``left_index`` and ``right_index`` keywords

Sometimes, rather than merging on a column, you would instead like to merge on an index.
For example, your data might look like this:

In [10]:
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')

#df1a
df2a

,hire_date
employee,
Lisa,2015
Bob,2018
Jake,2019
Sue,2017


You can use the index as the key for merging by specifying the ``left_index`` and ``right_index`` flags in ``pd.merge()``:

In [11]:
#display('df1a', 'df2a',
       # "pd.merge(df1a, df2a, left_index=True, right_index=True)")
pd.merge(df1a, df2a, left_index =True, right_index = True)

,dept,hire_date
employee,,
Bob,Accounting,2018
Jake,Engineering,2019
Lisa,Engineering,2015
Sue,HR,2017


For convenience, ``DataFrames`` implement the ``join()`` method, which performs a merge that defaults to joining on indices:

In [20]:
#display('df1a', 'df2a', 'df1a.join(df2a)')
df1a.join(df2a)

,dept,hire_date
employee,,
Bob,Accounting,2018
Jake,Engineering,2019
Lisa,Engineering,2015
Sue,HR,2017


If you'd like to mix indices and columns, you can combine ``left_index`` with ``right_on`` or ``left_on`` with ``right_index`` to get the desired behavior:

In [12]:
#display('df1a', 'df3', "pd.merge(df1a, df3, left_index=True, right_on='name')")
pd.merge(df1a, df3, left_index= True, right_on ='name')

,dept,name,salary
0,Accounting,Bob,70000
1,Engineering,Jake,80000
2,Engineering,Lisa,120000
3,HR,Sue,90000


### Merging on multiple columns

If two DataFrames have multiple common columns, specify them in a list using `on=[]`:

In [19]:
df3 = pd.DataFrame({
    'ID': [3, 4, 5, 6],
    'Subject': ['Math', 'Science', 'English', 'History'],
    'Grade': ['B', 'A', 'C', 'B'],
    'Year': [2021, 2021, 2022, 2022]
})

df4 = pd.DataFrame({
    'ID': [3, 4, 5, 6],
    'Year': [2021, 2021, 2022, 2022],
    'Teacher': ['Mr. Smith', 'Mrs. Johnson', 'Ms. Lee', 'Dr. Brown']
})

df_multi = pd.merge(df3, df4, on=['ID', 'Year'], how='inner')
print("\nMerge on Multiple Columns:\n", df_multi)



Merge on Multiple Columns:
    ID  Subject Grade  Year       Teacher
0   3     Math     B  2021     Mr. Smith
1   4  Science     A  2021  Mrs. Johnson
2   5  English     C  2022       Ms. Lee
3   6  History     B  2022     Dr. Brown


## Example: US States Data

Merge and join operations come up most often when combining data from different sources.
Here we will consider an example of some data about US states and their populations.

Let's take a look at the three datasets, using the Pandas ``read_csv()`` function:

In [24]:
pop = pd.read_csv('state-population.csv')
areas = pd.read_csv('state-areas.csv')
abbrevs = pd.read_csv('state-abbrevs.csv')


abbrevs.head()

,state,abbreviation
0,Alabama,AL
1,Alaska,AK
2,Arizona,AZ
3,Arkansas,AR
4,California,CA


In [15]:
areas.head()

,state,area (sq. mi)
0,Alabama,52423
1,Alaska,656425
2,Arizona,114006
3,Arkansas,53182
4,California,163707


In [26]:
pop.head()

,state/region,ages,year,population
0,AL,under18,2012,1117489.0
1,AL,total,2012,4817528.0
2,AL,under18,2010,1130966.0
3,AL,total,2010,4785570.0
4,AL,under18,2011,1125763.0


We want to compute a relatively straightforward result: rank US states by their 2010 population density.
We clearly have the data here to find this result, but we'll have to combine the datasets to find the result.

- start with a many-to-one merge that will give us the full state name within the population ``DataFrame``.
- merge based on the ``state/region``  column of ``pop``, and the ``abbreviation`` column of ``abbrevs``.
- use ``how='outer'`` to make sure no data is thrown away due to mismatched labels.

In [26]:
# how = 'outer'  keeps all rows from both data frames
merged = pd.merge(pop, abbrevs, how='outer', left_on='state/region', right_on='abbreviation')
merged = merged.drop('abbreviation', axis =1) # drop duplicate info
merged.head(10)
#merged.info()

,state/region,ages,year,population,state
0,AL,under18,2012,1117489.0,Alabama
1,AL,total,2012,4817528.0,Alabama
2,AL,under18,2010,1130966.0,Alabama
3,AL,total,2010,4785570.0,Alabama
4,AL,under18,2011,1125763.0,Alabama
5,AL,total,2011,4801627.0,Alabama
6,AL,total,2009,4757938.0,Alabama
7,AL,under18,2009,1134192.0,Alabama
8,AL,under18,2013,1111481.0,Alabama
9,AL,total,2013,4833722.0,Alabama


Let's double-check whether there were any mismatches here, which we can do by looking for rows with nulls:

In [29]:
merged.isnull().sum()

state/region     0
ages             0
year             0
population      20
state           96
dtype: int64

Some of the ``population`` and state info is null; let's tackle population which are they?

In [59]:
merged['population'].isnull()

0       False
1       False
2       False
3       False
4       False
        ...  
2539    False
2540    False
2541    False
2542    False
2543    False
Name: population, Length: 2544, dtype: bool

In [30]:
merged[merged['population'].isnull()].tail(20)
#merged['population'].isnull()


,state/region,ages,year,population,state
2448,PR,under18,1990,NaN,NaN
2449,PR,total,1990,NaN,NaN
2450,PR,total,1991,NaN,NaN
2451,PR,under18,1991,NaN,NaN
2452,PR,total,1993,NaN,NaN
2453,PR,under18,1993,NaN,NaN
2454,PR,under18,1992,NaN,NaN
2455,PR,total,1992,NaN,NaN
2456,PR,under18,1994,NaN,NaN
2457,PR,total,1994,NaN,NaN


It appears that all the null population values are from Puerto Rico prior to the year 2000; this is likely due to this data not being available from the original source.

More importantly, we see also that some of the new ``state`` entries are also null, which means that there was no corresponding entry in the ``abbrevs`` key!
Let's figure out which regions lack this match:

In [32]:
merged.loc[merged['state'].isnull(), 'state/region'].unique()

array(['PR', 'USA'], dtype=object)

The issue - population data includes entries for Puerto Rico (PR) and the United States as (USA). These entries do not appear in the state abbreviation data.
- Fix these quickly by filling in appropriate entries:

In [33]:
#featuretools
merged.loc[merged['state/region'] == 'PR', 'state'] = 'Puerto Rico'
merged.loc[merged['state/region'] == 'USA', 'state'] = 'United States'
merged.isnull().sum()

state/region     0
ages             0
year             0
population      20
state            0
dtype: int64

No more nulls in the ``state`` column: we're all set!

Now we can merge the result with the area data using a similar procedure.
Examining our results, we will want to join on the ``state`` column in both:

In [34]:
areas.head()

,state,area (sq. mi)
0,Alabama,52423
1,Alaska,656425
2,Arizona,114006
3,Arkansas,53182
4,California,163707


In [35]:
merged.head()

,state/region,ages,year,population,state
0,AL,under18,2012,1117489.0,Alabama
1,AL,total,2012,4817528.0,Alabama
2,AL,under18,2010,1130966.0,Alabama
3,AL,total,2010,4785570.0,Alabama
4,AL,under18,2011,1125763.0,Alabama


In [37]:
# how = 'left' includes all the rows of a data frame x and only those from y that match.
#areas.state
final = pd.merge(merged, areas, on='state', how='left')
final.head(15)

,state/region,ages,year,population,state,area (sq. mi)
0,AL,under18,2012,1117489.0,Alabama,52423.0
1,AL,total,2012,4817528.0,Alabama,52423.0
2,AL,under18,2010,1130966.0,Alabama,52423.0
3,AL,total,2010,4785570.0,Alabama,52423.0
4,AL,under18,2011,1125763.0,Alabama,52423.0
5,AL,total,2011,4801627.0,Alabama,52423.0
6,AL,total,2009,4757938.0,Alabama,52423.0
7,AL,under18,2009,1134192.0,Alabama,52423.0
8,AL,under18,2013,1111481.0,Alabama,52423.0
9,AL,total,2013,4833722.0,Alabama,52423.0


Again, let's check for nulls to see if there were any mismatches:

In [38]:
final.isnull().sum()

state/region      0
ages              0
year              0
population       20
state             0
area (sq. mi)    48
dtype: int64

There are nulls in the ``area`` column; we can take a look to see which regions were ignored here:

In [39]:
final['state'][final['area (sq. mi)'].isnull()].unique()

array(['United States'], dtype=object)

We see that our ``areas`` ``DataFrame`` does not contain the area of the United States as a whole.
We could insert the appropriate value (using the sum of all state areas, for instance), but in this case we'll just drop the null values because the population density of the entire United States is not relevant to our current discussion:

In [40]:
final.dropna(inplace=True) # transactional databases - commit
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AL,under18,2012,1117489.0,Alabama,52423.0
1,AL,total,2012,4817528.0,Alabama,52423.0
2,AL,under18,2010,1130966.0,Alabama,52423.0
3,AL,total,2010,4785570.0,Alabama,52423.0
4,AL,under18,2011,1125763.0,Alabama,52423.0


In [41]:
final.isnull().any()

state/region     False
ages             False
year             False
population       False
state            False
area (sq. mi)    False
dtype: bool

Now we have all the data we need. To answer the question of interest, let's first select the portion of the data corresponding with the year 2000, and the total population.
We'll use the ``query()`` function.

In [42]:
data2000 = final.query("year == 2000 & ages == 'total'")
data2000.head(20)

,state/region,ages,year,population,state,area (sq. mi)
28,AL,total,2000,4452173.0,Alabama,52423.0
68,AK,total,2000,627963.0,Alaska,656425.0
124,AZ,total,2000,5160586.0,Arizona,114006.0
162,AR,total,2000,2678588.0,Arkansas,53182.0
220,CA,total,2000,33987977.0,California,163707.0
258,CO,total,2000,4326921.0,Colorado,104100.0
316,CT,total,2000,3411777.0,Connecticut,5544.0
354,DE,total,2000,786373.0,Delaware,1954.0
412,DC,total,2000,572046.0,District of Columbia,68.0
452,FL,total,2000,16047515.0,Florida,65758.0


We compute the population density and display it in order.
- Start by re-indexing the data on the state, and then compute the result:

In [43]:
#data2000.info()
data2000.set_index('state', inplace=True)
density = data2000['population'] / data2000['area (sq. mi)']

In [44]:
density.sort_values(ascending=False, inplace=True)
density.head()

state
District of Columbia    8412.441176
Puerto Rico             1084.098151
New Jersey               966.592639
Rhode Island             679.785113
Connecticut              615.399892
dtype: float64

The result is a ranking of US states in order of their 2000 population density, in residents per square mile.
The densest region in this dataset is Washington, DC (i.e., the District of Columbia); among states, the densest is New Jersey.

We can also check the end of the list:

In [45]:
#density.tail(20)

density.to_csv('density.csv')

In [46]:
density = pd.DataFrame(density, columns = ['Density'])
density.to_csv('density.csv')

We see that the least dense state, by far, is Alaska, averaging slightly over one resident per square mile.

This type of messy data merging is a common task when trying to answer questions using real-world data sources.